# Dataset Preparation (UPDATED) — Member 2: Zainib
**Project:** Depression Detection using Machine Learning Techniques

### What changed from the original version
The original split used `question1`–`question9` (PHQ-9 items) as features to predict `Distressed`, which is mathematically derived from those same items — that's data leakage (confirmed in the updated Phase 3 notebook).

This version uses the leakage-free feature set produced by the updated Phase 3 preprocessing (`cleaned_features.csv`): demographics + GAD-7 (anxiety) + ISI (insomnia) + PSS (stress) items. `Distressed` remains the target.

Two changes to the split itself:
- **Stratified split** (`stratify=y`) — required because the classes are imbalanced (~5.4% Distressed). Without stratification, a random split could easily under- or over-represent the minority class in train or test.
- Same **80:20 ratio** and `random_state=42`, kept consistent with the team's original convention.

In [1]:
# Member 2 Zainib - Dataset Preparation (UPDATED, leakage-free features)
# Depression Detection Project

# Import Libraries
import pandas as pd
from sklearn.model_selection import train_test_split

# Read Dataset
df = pd.read_csv("cleaned_features.csv")

# Display Basic Information
print("First Five Rows")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

First Five Rows
   Distressed   age  gad7_q1  gad7_q2  gad7_q3  gad7_q4  gad7_q5  gad7_q6  \
0           0  19.0        0        0        0        0        0        0   
1           0  18.0        0        0        0        0        0        0   
2           0  40.0        0        0        0        0        0        0   
3           0  23.0        0        0        0        0        0        0   
4           0  21.0        0        0        0        0        0        0   

   gad7_q7  isi_q1  ...  gender_male  edu_bachelor's degree  \
0        0       0  ...        False                   True   
1        0       0  ...        False                   True   
2        0       0  ...         True                  False   
3        0       0  ...        False                   True   
4        0       0  ...        False                   True   

   edu_doctorate degree  edu_master's degree  \
0                 False                False   
1                 False                False  

## Define Input Features and Target
Input features are everything except `Distressed` — that is, demographics (age, gender, education, smoking, drinking) plus all GAD-7, ISI, and PSS question responses. No PHQ-9-derived column is present.

In [2]:
# Input Features (everything except the label)
feature_cols = [c for c in df.columns if c != "Distressed"]
X = df[feature_cols]

# Target Variable
y = df["Distressed"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)
print("\nFeature columns:")
print(feature_cols)

Features Shape: (24292, 39)
Target Shape: (24292,)

Feature columns:
['age', 'gad7_q1', 'gad7_q2', 'gad7_q3', 'gad7_q4', 'gad7_q5', 'gad7_q6', 'gad7_q7', 'isi_q1', 'isi_q2', 'isi_q3', 'isi_q4', 'isi_q5', 'isi_q6', 'isi_q7', 'pss_q1', 'pss_q2', 'pss_q3', 'pss_q4', 'pss_q5', 'pss_q6', 'pss_q7', 'pss_q8', 'pss_q9', 'pss_q10', 'pss_q11', 'pss_q12', 'pss_q13', 'pss_q14', 'gender_male', "edu_bachelor's degree", 'edu_doctorate degree', "edu_master's degree", 'smoke_former smoker (cumulative smoking >10 packs), but not in the past year', 'smoke_never smokes', 'smoke_occasional smoker (cumulative smoking <10 packs)', 'drink_drank in the past (more than once a week), but not in the past year', 'drink_drinks occasionally (less than once a week)', 'drink_never drinks']


## Check Class Distribution (before split)

In [3]:
print("Class distribution (counts):")
print(y.value_counts())

print("\nClass distribution (%):")
print((y.value_counts(normalize=True) * 100).round(2))

Class distribution (counts):
Distressed
0    22976
1     1316
Name: count, dtype: int64

Class distribution (%):
Distressed
0    94.58
1     5.42
Name: proportion, dtype: float64


## 80:20 Stratified Train-Test Split
`stratify=y` keeps the ~5.4% Distressed ratio consistent in both the training and test sets — important given the class imbalance.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (19433, 39)
X_test shape: (4859, 39)
y_train shape: (19433,)
y_test shape: (4859,)


## Verify Class Distribution is Preserved in Both Splits

In [5]:
print("Train set class distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTest set class distribution (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

Train set class distribution (%):
Distressed
0    94.58
1     5.42
Name: proportion, dtype: float64

Test set class distribution (%):
Distressed
0    94.59
1     5.41
Name: proportion, dtype: float64


## Dataset Summary

| Split | Rows | Distressed (%) |
|---|---|---|
| Train | 19,433 | ~5.42% |
| Test | 4,859 | ~5.42% |

- 40 columns total in `cleaned_features.csv`: 1 label (`Distressed`) + 39 features (age, gender, education, smoking, drinking [one-hot encoded], 7 GAD-7 items, 7 ISI items, 14 PSS items)
- No PHQ-9 question or score column is present anywhere in the feature set — this is confirmed leakage-free
- Class imbalance (~5.4% positive) is preserved identically across train and test thanks to stratification
- **Important note for the whole team:** because of this imbalance, Accuracy alone is not a meaningful metric — a model predicting "Non-Distressed" for everyone would already score ~94.6% accuracy. Precision, Recall, F1-Score, and ROC-AUC (already required deliverables) are what actually show whether a model is learning anything real.

## Save Final Training and Testing Datasets

In [6]:
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("All files saved successfully.")
print("X_train.csv, X_test.csv, y_train.csv, y_test.csv are ready for Phase 4 model development.")

All files saved successfully.
X_train.csv, X_test.csv, y_train.csv, y_test.csv are ready for Phase 4 model development.
